In [1]:
import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Pandas: 3.0.6
NumPy: 2.5.4
Scikit-learn: 1.9.1


In [2]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent

train_path = ROOT_DIR / "dataset" / "UNSW_NB15_training-set.csv"
test_path = ROOT_DIR / "dataset" / "UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (175341, 45)
Testing shape: (82332, 45)


In [3]:
train_df.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  str    
 3   service            175341 non-null  str    
 4   state              175341 non-null  str    
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  sinpkt       

In [5]:
train_df["label"].value_counts()

label
1    119341
0     56000
Name: count, dtype: int64

In [6]:
train_df["attack_cat"].value_counts(dropna=False)

attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64

In [7]:
train_df.isnull().sum().sort_values(ascending=False).head(15)

id         0
dur        0
proto      0
service    0
state      0
spkts      0
dpkts      0
sbytes     0
dbytes     0
rate       0
sttl       0
dttl       0
sload      0
dload      0
sloss      0
dtype: int64

In [8]:
print("1. DATASET SHAPE")
print(train_df.shape)

print("\n2. COLUMN NAMES")
print(train_df.columns.tolist())

print("\n3. LABEL DISTRIBUTION")
print(train_df["label"].value_counts())

print("\n4. DATA TYPES")
print(train_df.dtypes)

print("\n5. MISSING VALUES")
print(train_df.isnull().sum().sort_values(ascending=False).head(15))

1. DATASET SHAPE
(175341, 45)

2. COLUMN NAMES
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

3. LABEL DISTRIBUTION
label
1    119341
0     56000
Name: count, dtype: int64

4. DATA TYPES
id                     int64
dur                  float64
proto                    str
service                  str
state                    str
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   

In [9]:
print(train_df.dtypes.value_counts())

int64      30
float64    11
str         4
Name: count, dtype: int64


In [10]:
categorical_cols = train_df.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['proto', 'service', 'state', 'attack_cat']


In [11]:
missing = train_df.isnull().sum()

print("Total missing values:", missing.sum())

print("\nColumns with missing values:")
print(missing[missing > 0])


Total missing values: 0

Columns with missing values:
Series([], dtype: int64)


In [12]:
print(
    "Duplicate rows:",
    train_df.duplicated().sum()
)

Duplicate rows: 0


In [13]:
for col in ["proto", "service", "state"]:
    print(f"\n{col}:")
    print(train_df[col].value_counts().head(10))


proto:
proto
tcp       79946
udp       63283
unas      12084
arp        2859
ospf       2595
sctp       1150
any         300
gre         225
sun-nd      201
swipe       201
Name: count, dtype: int64

service:
service
-           94168
dns         47294
http        18724
smtp         5058
ftp-data     3995
ftp          3428
ssh          1302
pop3         1105
dhcp           94
snmp           80
Name: count, dtype: int64

state:
state
INT    82275
FIN    77825
CON    13152
REQ     1991
RST       83
ECO       12
PAR        1
URN        1
no         1
Name: count, dtype: int64


In [14]:
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

Training shape: (175341, 45)
Testing shape: (82332, 45)

Training labels:
label
1    119341
0     56000
Name: count, dtype: int64

Testing labels:
label
1    45332
0    37000
Name: count, dtype: int64


In [16]:
# Check feature types
print("Data types:")
print(train_df.dtypes.value_counts())

# Check categorical columns
categorical_cols = ["proto", "service", "state"]

print("\nUnique categorical values:")
for col in categorical_cols:
    print(f"{col}: {train_df[col].nunique()}")

# Verify labels
print("\nUnique training labels:")
print(sorted(train_df["label"].unique()))

print("\nUnique testing labels:")
print(sorted(test_df["label"].unique()))

# Check missing values in test data
print("\nMissing values in test data:")
print(test_df.isnull().sum().sum())

# Check duplicates in test data
print("\nDuplicate rows in test data:")
print(test_df.duplicated().sum())

Data types:
int64      30
float64    11
str         4
Name: count, dtype: int64

Unique categorical values:
proto: 133
service: 13
state: 9

Unique training labels:
[np.int64(0), np.int64(1)]

Unique testing labels:
[np.int64(0), np.int64(1)]

Missing values in test data:
0

Duplicate rows in test data:
0
